In [1]:
# =============================================================================
# PHASE 5: PSI-BASED ADAPTIVE RETRAINING
# Intelligent Adaptive Credit Risk Scoring Model
# Concept drift detection using Population Stability Index (PSI)
# Drift simulated via controlled perturbation of holdout feature distributions
# Triggers automatic model retraining when PSI > 0.25
# Implements semantic model versioning for audit trail
# =============================================================================

import pandas as pd
import numpy as np
import joblib
import os
import json
import warnings
import time
import shutil
from datetime import datetime
warnings.filterwarnings('ignore')

from sklearn.linear_model    import LogisticRegression
from sklearn.tree            import DecisionTreeClassifier
from sklearn.ensemble        import RandomForestClassifier
from xgboost                 import XGBClassifier

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics         import (roc_auc_score, recall_score,
                                     precision_score, f1_score,
                                     accuracy_score, matthews_corrcoef)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# =============================================================================
# STEP 1: LOAD ARTEFACTS
# =============================================================================

print("=" * 65)
print("PHASE 5: PSI-BASED ADAPTIVE RETRAINING")
print("=" * 65)
print("\nSTEP 1: Loading artefacts...")

required_files = [
    'artefacts/X_train.csv',
    'artefacts/X_test.csv',
    'artefacts/y_train.csv',
    'artefacts/y_test.csv',
    'artefacts/best_model.pkl',
    'artefacts/best_model_name.pkl',
    'artefacts/class_weights.pkl'
]
missing = [f for f in required_files if not os.path.exists(f)]
if missing:
    raise FileNotFoundError(
        f"\n  ERROR: Missing artefacts:\n"
        + "\n".join(f"    - {f}" for f in missing)
        + "\n  Please run phase2_model_training.py first."
    )

X_train         = pd.read_csv('artefacts/X_train.csv')
X_test          = pd.read_csv('artefacts/X_test.csv')
y_train         = pd.read_csv('artefacts/y_train.csv').squeeze()
y_test          = pd.read_csv('artefacts/y_test.csv').squeeze()
best_model      = joblib.load('artefacts/best_model.pkl')
best_model_name = joblib.load('artefacts/best_model_name.pkl')
class_weight_dict = joblib.load('artefacts/class_weights.pkl')

print(f"  X_train      : {X_train.shape}")
print(f"  X_test       : {X_test.shape}")
print(f"  Best Model   : {best_model_name}")

os.makedirs('artefacts/psi',         exist_ok=True)
os.makedirs('artefacts/versions',    exist_ok=True)
os.makedirs('plots/psi',             exist_ok=True)

print("\n  Output directories ready:")
print("    artefacts/psi/")
print("    artefacts/versions/")
print("    plots/psi/")

# =============================================================================
# STEP 2: PSI FUNCTION DEFINITIONS
# =============================================================================

print("\n" + "=" * 65)
print("STEP 2: PSI Function Definitions")
print("=" * 65)

def compute_psi_feature(expected, actual, bins=10):
    """
    Compute PSI for a single feature.

    Parameters
    ----------
    expected : array-like — reference distribution (training)
    actual   : array-like — current distribution (new data)
    bins     : int        — number of quantile bins

    Returns
    -------
    psi   : float — PSI value
    table : pd.DataFrame — bin-level breakdown
    """
    # Define bin edges from the expected (reference) distribution
    breakpoints = np.nanpercentile(expected,
                                   np.linspace(0, 100, bins + 1))
    breakpoints  = np.unique(breakpoints)  # remove duplicates

    # Bin both distributions
    exp_counts, _ = np.histogram(expected, bins=breakpoints)
    act_counts, _ = np.histogram(actual,   bins=breakpoints)

    # Convert to proportions — add small epsilon to avoid log(0)
    eps     = 1e-6
    exp_pct = (exp_counts / len(expected)) + eps
    act_pct = (act_counts / len(actual))   + eps

    # PSI formula
    psi_bins = (act_pct - exp_pct) * np.log(act_pct / exp_pct)
    psi      = psi_bins.sum()

    table = pd.DataFrame({
        'Bin':         range(1, len(psi_bins) + 1),
        'Expected_%':  np.round(exp_pct * 100, 2),
        'Actual_%':    np.round(act_pct * 100, 2),
        'PSI_contrib': np.round(psi_bins, 6)
    })
    return psi, table


def compute_psi_score(expected_probs, actual_probs, bins=10):
    """
    Compute PSI on predicted probability distributions.
    This captures overall model behaviour shift — more informative
    than per-feature PSI for credit scoring applications.
    """
    psi, table = compute_psi_feature(expected_probs, actual_probs, bins)
    return psi, table


def interpret_psi(psi):
    """Return PSI band and recommended action."""
    if psi < 0.10:
        return "Stable",   "No action required — model is stable."
    elif psi < 0.25:
        return "Moderate", "Monitor closely — investigate feature shifts."
    else:
        return "Critical", "Trigger retraining — significant concept drift detected."


print("  compute_psi_feature() — per-feature PSI")
print("  compute_psi_score()   — probability distribution PSI")
print("  interpret_psi()       — PSI band + recommended action")
print("\n  PSI Thresholds:")
print("    < 0.10  → Stable   (no action)")
print("    0.10–0.25 → Moderate (monitor)")
print("    > 0.25  → Critical (retrain)")

# =============================================================================
# STEP 3: SIMULATE CONCEPT DRIFT
# Controlled perturbation of X_test to simulate new incoming data
# Three drift scenarios: None, Moderate, Severe
# =============================================================================

print("\n" + "=" * 65)
print("STEP 3: Simulating Concept Drift")
print("=" * 65)

np.random.seed(42)

def simulate_drift(X, drift_level='moderate'):
    """
    Simulate concept drift by perturbing feature distributions.

    drift_level:
      'none'     — no perturbation (baseline)
      'moderate' — mild shift, PSI ~ 0.10–0.25
      'severe'   — strong shift, PSI > 0.25
    """
    X_drifted = X.copy()

    if drift_level == 'none':
        return X_drifted

    numeric_cols = X_drifted.select_dtypes(include=[np.number]).columns

    for col in numeric_cols:
        col_std  = X_drifted[col].std()
        col_mean = X_drifted[col].mean()

        if drift_level == 'moderate':
            # Shift mean by 0.5 std, add small noise
            X_drifted[col] = (X_drifted[col]
                              + 0.5 * col_std
                              + np.random.normal(0, 0.1 * col_std,
                                                 len(X_drifted)))
        elif drift_level == 'severe':
            # Shift mean by 1.5 std, add larger noise, flip some values
            X_drifted[col] = (X_drifted[col]
                              + 1.5 * col_std
                              + np.random.normal(0, 0.3 * col_std,
                                                 len(X_drifted)))

    return X_drifted

# Generate three drift scenarios
X_no_drift       = simulate_drift(X_test, 'none')
X_moderate_drift = simulate_drift(X_test, 'moderate')
X_severe_drift   = simulate_drift(X_test, 'severe')

print(f"  Scenario 1: No drift       — X_test unchanged")
print(f"  Scenario 2: Moderate drift — mean shifted by 0.5σ + noise")
print(f"  Scenario 3: Severe drift   — mean shifted by 1.5σ + noise")

# =============================================================================
# STEP 4: COMPUTE PSI — PROBABILITY DISTRIBUTION
# Compare predicted probability distributions:
# Reference = model on X_train
# Current   = model on drifted X_test
# =============================================================================

print("\n" + "=" * 65)
print("STEP 4: PSI Computation — Probability Distributions")
print("=" * 65)

# Reference: predicted probabilities on training set
ref_probs = best_model.predict_proba(X_train)[:, 1]

scenarios = {
    'No Drift':       X_no_drift,
    'Moderate Drift': X_moderate_drift,
    'Severe Drift':   X_severe_drift
}

psi_results = {}

print(f"\n  {'Scenario':<20} {'PSI':>8} {'Band':<12} Action")
print("  " + "-" * 70)

for scenario_name, X_scenario in scenarios.items():
    cur_probs        = best_model.predict_proba(X_scenario)[:, 1]
    psi, psi_table   = compute_psi_score(ref_probs, cur_probs, bins=10)
    band, action     = interpret_psi(psi)

    psi_results[scenario_name] = {
        'PSI':    round(psi, 4),
        'Band':   band,
        'Action': action,
        'ref_probs': ref_probs,
        'cur_probs': cur_probs
    }

    print(f"  {scenario_name:<20} {psi:>8.4f} {band:<12} {action}")

    # Save PSI table per scenario
    fname = scenario_name.lower().replace(' ', '_')
    psi_table.to_csv(f'artefacts/psi/psi_table_{fname}.csv', index=False)

with open('artefacts/psi/psi_summary.json', 'w') as f:
    json.dump({k: {kk: vv for kk, vv in v.items()
                   if kk not in ['ref_probs','cur_probs']}
               for k, v in psi_results.items()}, f, indent=4)
print(f"\n  Saved: artefacts/psi/psi_summary.json")

# =============================================================================
# STEP 5: PSI VISUALISATION
# =============================================================================

print("\n" + "=" * 65)
print("STEP 5: PSI Visualisation")
print("=" * 65)

# Plot 1: Probability distribution comparison across all scenarios
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)
colors_scenario = ['#2ca02c', '#ff7f0e', '#d62728']

for ax, (scenario_name, res), color in zip(
        axes, psi_results.items(), colors_scenario):
    ax.hist(res['ref_probs'], bins=20, alpha=0.6,
            color='#1f77b4', label='Reference (Train)')
    ax.hist(res['cur_probs'], bins=20, alpha=0.6,
            color=color, label=f'Current ({scenario_name})')
    ax.set_title(f"{scenario_name}\nPSI = {res['PSI']:.4f}  [{res['Band']}]",
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicted Probability', fontsize=10)
    ax.set_ylabel('Count', fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    ax.axvline(x=0.5, color='black', linestyle='--', lw=1)

plt.suptitle('Predicted Probability Distributions — Drift Scenarios',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/psi/probability_distributions.png',
            dpi=150, bbox_inches='tight')
plt.close()
print("  Saved: plots/psi/probability_distributions.png")

# Plot 2: PSI bar chart with threshold lines
fig, ax = plt.subplots(figsize=(9, 5))
scenario_names = list(psi_results.keys())
psi_vals       = [psi_results[s]['PSI'] for s in scenario_names]
bar_colors     = ['#2ca02c', '#ff7f0e', '#d62728']

bars = ax.bar(scenario_names, psi_vals, color=bar_colors,
              alpha=0.85, edgecolor='black', linewidth=0.5, width=0.5)
for bar, val in zip(bars, psi_vals):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.005,
            f'{val:.4f}', ha='center', va='bottom',
            fontsize=11, fontweight='bold')

ax.axhline(y=0.10, color='#ff7f0e', linestyle='--', lw=1.5,
           label='Moderate threshold (0.10)')
ax.axhline(y=0.25, color='#d62728', linestyle='--', lw=1.5,
           label='Critical threshold (0.25)')
ax.set_ylabel('PSI Value', fontsize=12)
ax.set_title('Population Stability Index — Drift Scenarios',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, max(psi_vals) * 1.25)
plt.tight_layout()
plt.savefig('plots/psi/psi_bar_chart.png', dpi=150, bbox_inches='tight')
plt.close()
print("  Saved: plots/psi/psi_bar_chart.png")

# =============================================================================
# STEP 6: FEATURE-LEVEL PSI — SEVERE DRIFT SCENARIO
# Identify which features drifted most
# =============================================================================

print("\n" + "=" * 65)
print("STEP 6: Feature-Level PSI Analysis (Severe Drift)")
print("=" * 65)

feature_psi = {}
for col in X_train.columns:
    psi_val, _ = compute_psi_feature(
        X_train[col].values,
        X_severe_drift[col].values,
        bins=10
    )
    feature_psi[col] = round(psi_val, 4)

feature_psi_df = pd.DataFrame({
    'Feature': list(feature_psi.keys()),
    'PSI':     list(feature_psi.values())
}).sort_values('PSI', ascending=False).reset_index(drop=True)

feature_psi_df['Band'] = feature_psi_df['PSI'].apply(
    lambda x: interpret_psi(x)[0])

print(f"\n  {'Feature':<35} {'PSI':>8} {'Band':<12}")
print("  " + "-" * 57)
for _, row in feature_psi_df.head(10).iterrows():
    print(f"  {row['Feature']:<35} {row['PSI']:>8.4f} {row['Band']:<12}")

feature_psi_df.to_csv('artefacts/psi/feature_psi_severe.csv', index=False)
print(f"\n  Saved: artefacts/psi/feature_psi_severe.csv")

# Plot feature PSI
fig, ax = plt.subplots(figsize=(12, 7))
top_features = feature_psi_df.head(15)
bar_cols = top_features['Band'].map(
    {'Stable': '#2ca02c', 'Moderate': '#ff7f0e', 'Critical': '#d62728'})
bars = ax.barh(top_features['Feature'][::-1],
               top_features['PSI'][::-1],
               color=bar_cols[::-1],
               alpha=0.85, edgecolor='black', linewidth=0.5)
ax.axvline(x=0.10, color='#ff7f0e', linestyle='--', lw=1.5,
           label='Moderate (0.10)')
ax.axvline(x=0.25, color='#d62728', linestyle='--', lw=1.5,
           label='Critical (0.25)')
ax.set_xlabel('PSI Value', fontsize=12)
ax.set_title('Feature-Level PSI — Severe Drift Scenario\n'
             '(Green=Stable, Orange=Moderate, Red=Critical)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('plots/psi/feature_psi_chart.png', dpi=150, bbox_inches='tight')
plt.close()
print("  Saved: plots/psi/feature_psi_chart.png")

# =============================================================================
# STEP 7: ADAPTIVE RETRAINING — TRIGGERED BY SEVERE DRIFT
# Retrains XGBoost on combined train + drifted data
# Compares pre- and post-retrain performance
# =============================================================================

print("\n" + "=" * 65)
print("STEP 7: Adaptive Retraining (Triggered by Severe Drift)")
print("=" * 65)

severe_psi  = psi_results['Severe Drift']['PSI']
band, action = interpret_psi(severe_psi)

print(f"\n  PSI (Severe Drift) : {severe_psi:.4f}")
print(f"  Band               : {band}")
print(f"  Action             : {action}")

if severe_psi >= 0.25:
    print(f"\n  ✓ PSI threshold exceeded — initiating retraining...")

    # Combine original training data with drifted data
    X_retrain = pd.concat([X_train, X_severe_drift], ignore_index=True)
    y_retrain = pd.concat([y_train, y_test],          ignore_index=True)

    print(f"  Retraining data : {X_retrain.shape}  "
          f"(original {X_train.shape[0]} + drifted {X_severe_drift.shape[0]})")

    # Retrain XGBoost with RandomizedSearchCV
    base_spw = class_weight_dict[1] / class_weight_dict[0]
    xgb_spw  = round(base_spw * 1.5, 4)

    retrain_model = XGBClassifier(
        scale_pos_weight = xgb_spw,
        random_state     = 42,
        eval_metric      = 'logloss',
        verbosity        = 0
    )

    retrain_param_dist = {
        'n_estimators':     [300, 400, 500],
        'max_depth':        [3, 4, 5],
        'learning_rate':    [0.01, 0.03, 0.05],
        'subsample':        [0.7, 0.8, 0.9],
        'colsample_bytree': [0.6, 0.7, 0.8],
        'min_child_weight': [1, 2, 3],
        'gamma':            [0, 0.05, 0.1],
        'reg_alpha':        [0, 0.01],
        'reg_lambda':       [1, 2]
    }

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    t0  = time.time()

    retrain_search = RandomizedSearchCV(
        estimator           = retrain_model,
        param_distributions = retrain_param_dist,
        n_iter              = 50,
        scoring             = 'roc_auc',
        cv                  = skf,
        n_jobs              = -1,
        verbose             = 0,
        refit               = True,
        random_state        = 42
    )
    retrain_search.fit(X_retrain, y_retrain)
    elapsed = time.time() - t0

    retrained_model = retrain_search.best_estimator_
    print(f"\n  Retraining complete in {elapsed:.1f}s")
    print(f"  Best CV ROC-AUC : {retrain_search.best_score_:.4f}")
    print(f"  Best params     : {retrain_search.best_params_}")

    # ==========================================================================
    # STEP 8: PRE vs POST RETRAIN PERFORMANCE COMPARISON
    # ==========================================================================

    print("\n" + "=" * 65)
    print("STEP 8: Pre vs Post Retrain Performance")
    print("=" * 65)

    def evaluate_model(model, X, y, label):
        y_prob = model.predict_proba(X)[:, 1]
        y_pred = model.predict(X)
        return {
            'Model':     label,
            'ROC-AUC':   round(roc_auc_score(y, y_prob),        4),
            'Recall':    round(recall_score(y, y_pred),          4),
            'Precision': round(precision_score(y, y_pred,
                               zero_division=0),                 4),
            'F1':        round(f1_score(y, y_pred),              4),
            'Accuracy':  round(accuracy_score(y, y_pred),        4),
            'MCC':       round(matthews_corrcoef(y, y_pred),      4)
        }

    pre_metrics  = evaluate_model(best_model,      X_test, y_test,
                                  f'Pre-Retrain  ({best_model_name} v1.0)')
    post_metrics = evaluate_model(retrained_model, X_test, y_test,
                                  f'Post-Retrain ({best_model_name} v2.0)')

    comp_df = pd.DataFrame([pre_metrics, post_metrics])
    print(f"\n  {comp_df.to_string(index=False)}")

    # Delta
    print(f"\n  Performance Delta (Post - Pre):")
    for col in ['ROC-AUC','Recall','Precision','F1','Accuracy','MCC']:
        delta = post_metrics[col] - pre_metrics[col]
        arrow = '↑' if delta > 0 else ('↓' if delta < 0 else '→')
        print(f"    {col:<12}: {delta:+.4f}  {arrow}")

    comp_df.to_csv('artefacts/psi/retrain_comparison.csv', index=False)
    print(f"\n  Saved: artefacts/psi/retrain_comparison.csv")

    # Plot comparison
    metrics_cols = ['ROC-AUC','Recall','Precision','F1','Accuracy','MCC']
    x     = np.arange(len(metrics_cols))
    width = 0.35

    fig, ax = plt.subplots(figsize=(13, 6))
    bars1 = ax.bar(x - width/2,
                   [pre_metrics[m]  for m in metrics_cols],
                   width, label='Pre-Retrain  (v1.0)',
                   color='#1f77b4', alpha=0.85,
                   edgecolor='black', linewidth=0.5)
    bars2 = ax.bar(x + width/2,
                   [post_metrics[m] for m in metrics_cols],
                   width, label='Post-Retrain (v2.0)',
                   color='#d62728', alpha=0.85,
                   edgecolor='black', linewidth=0.5)

    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.005,
                f'{bar.get_height():.3f}',
                ha='center', va='bottom', fontsize=8)
    for bar in bars2:
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.005,
                f'{bar.get_height():.3f}',
                ha='center', va='bottom', fontsize=8)

    ax.set_xticks(x)
    ax.set_xticklabels(metrics_cols, fontsize=11)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel('Score', fontsize=12)
    ax.set_title(f'Pre vs Post Retraining — {best_model_name}\n'
                 f'(Triggered by PSI = {severe_psi:.4f} > 0.25)',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(axis='y', alpha=0.3)
    ax.axhline(y=0.7, color='gray', linestyle='--', lw=0.8, alpha=0.6)
    plt.tight_layout()
    plt.savefig('plots/psi/retrain_comparison.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("  Saved: plots/psi/retrain_comparison.png")

    # ==========================================================================
    # STEP 9: SEMANTIC MODEL VERSIONING
    # ==========================================================================

    print("\n" + "=" * 65)
    print("STEP 9: Semantic Model Versioning")
    print("=" * 65)

    timestamp  = datetime.now().strftime("%Y%m%d_%H%M%S")
    version_v1 = "v1.0.0"
    version_v2 = "v2.0.0"

    # Save retrained model
    retrain_path = f'artefacts/versions/{best_model_name.lower().replace(" ","_")}_{version_v2}.pkl'
    joblib.dump(retrained_model, retrain_path)

    # Version registry
    version_registry = {
        version_v1: {
            "version":      version_v1,
            "timestamp":    "Original training — Phase 2",
            "trigger":      "Initial training",
            "PSI_at_train": None,
            "model_file":   f'artefacts/models/xgboost/xgboost.pkl',
            "metrics":      {k: pre_metrics[k]
                             for k in ['ROC-AUC','Recall',
                                       'Precision','F1','Accuracy','MCC']}
        },
        version_v2: {
            "version":      version_v2,
            "timestamp":    timestamp,
            "trigger":      f"PSI = {severe_psi:.4f} exceeded critical "
                            f"threshold (0.25) — Severe drift detected",
            "PSI_at_train": severe_psi,
            "model_file":   retrain_path,
            "metrics":      {k: post_metrics[k]
                             for k in ['ROC-AUC','Recall',
                                       'Precision','F1','Accuracy','MCC']},
            "best_params":  {k: (int(v) if isinstance(v, np.integer) else v)
                             for k, v in retrain_search.best_params_.items()}
        }
    }

    with open('artefacts/versions/version_registry.json', 'w') as f:
        json.dump(version_registry, f, indent=4)

    print(f"\n  Version Registry:")
    for ver, info in version_registry.items():
        print(f"\n    [{ver}]")
        print(f"      Trigger   : {info['trigger']}")
        print(f"      ROC-AUC   : {info['metrics']['ROC-AUC']}")
        print(f"      Recall    : {info['metrics']['Recall']}")
        print(f"      Model file: {info['model_file']}")

    print(f"\n  Saved: artefacts/versions/version_registry.json")
    print(f"  Saved: {retrain_path}")

    # Update best model to retrained version
    joblib.dump(retrained_model, 'artefacts/best_model.pkl')
    print(f"  Updated: artefacts/best_model.pkl → {version_v2}")

else:
    print(f"\n  PSI below threshold — no retraining required.")
    print(f"  Current model remains active.")

# =============================================================================
# STEP 10: PSI MONITORING LOG
# Simulates what a production monitoring log would look like
# =============================================================================

print("\n" + "=" * 65)
print("STEP 10: PSI Monitoring Log")
print("=" * 65)

monitoring_log = []
for scenario_name, res in psi_results.items():
    band, action = interpret_psi(res['PSI'])
    monitoring_log.append({
        'Timestamp':  datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        'Scenario':   scenario_name,
        'PSI':        res['PSI'],
        'Band':       band,
        'Action':     action,
        'Retrained':  'Yes' if res['PSI'] >= 0.25 else 'No'
    })

log_df = pd.DataFrame(monitoring_log)
log_df.to_csv('artefacts/psi/monitoring_log.csv', index=False)

print(f"\n  {'Scenario':<20} {'PSI':>8} {'Band':<12} {'Retrained':>10}")
print("  " + "-" * 54)
for _, row in log_df.iterrows():
    print(f"  {row['Scenario']:<20} {row['PSI']:>8.4f} "
          f"{row['Band']:<12} {row['Retrained']:>10}")

print(f"\n  Saved: artefacts/psi/monitoring_log.csv")

# =============================================================================
# FINAL SUMMARY
# =============================================================================

print("\n" + "=" * 65)
print("PHASE 5 COMPLETE — PSI ADAPTIVE RETRAINING SUMMARY")
print("=" * 65)

print(f"\n  PSI Results:")
for scenario, res in psi_results.items():
    band, _ = interpret_psi(res['PSI'])
    print(f"    {scenario:<20} PSI={res['PSI']:.4f}  [{band}]")

print(f"\n  Retraining : Triggered at PSI={severe_psi:.4f} (Severe Drift)")
print(f"  Versioning : v1.0.0 → v2.0.0")

print(f"\n  Artefacts saved to artefacts/psi/:")
for f in sorted(os.listdir('artefacts/psi')):
    print(f"    ├── {f}")

print(f"\n  Plots saved to plots/psi/:")
for f in sorted(os.listdir('plots/psi')):
    print(f"    ├── {f}")

print(f"\n  All five phases complete.")
print(f"  Ready for Streamlit application development.")


PHASE 5: PSI-BASED ADAPTIVE RETRAINING

STEP 1: Loading artefacts...
  X_train      : (800, 20)
  X_test       : (200, 20)
  Best Model   : XGBoost

  Output directories ready:
    artefacts/psi/
    artefacts/versions/
    plots/psi/

STEP 2: PSI Function Definitions
  compute_psi_feature() — per-feature PSI
  compute_psi_score()   — probability distribution PSI
  interpret_psi()       — PSI band + recommended action

  PSI Thresholds:
    < 0.10  → Stable   (no action)
    0.10–0.25 → Moderate (monitor)
    > 0.25  → Critical (retrain)

STEP 3: Simulating Concept Drift
  Scenario 1: No drift       — X_test unchanged
  Scenario 2: Moderate drift — mean shifted by 0.5σ + noise
  Scenario 3: Severe drift   — mean shifted by 1.5σ + noise

STEP 4: PSI Computation — Probability Distributions

  Scenario                  PSI Band         Action
  ----------------------------------------------------------------------
  No Drift               0.0992 Stable       No action required — model is 

In [2]:
import pandas as pd
import os

os.chdir(r"C:\Users\lenovo\Desktop\Credit Scoring Model APK\all scripts")

X_train = pd.read_csv("artefacts/X_train.csv")
print(X_train.columns.tolist())
print(X_train.shape)

['checkingstatus1', 'duration', 'history', 'purpose', 'amount', 'savings', 'employ', 'installment', 'status', 'others', 'residence', 'property', 'age', 'otherplans', 'housing', 'cards', 'job', 'liable', 'tele', 'foreign']
(800, 20)


In [3]:
print(X_train.head(2))
print(X_train.describe())

   checkingstatus1  duration  history  purpose    amount  savings  employ  \
0                0  1.289592        2        1  1.925766        4       4   
1                3 -0.742595        2        4 -0.892853        0       4   

   installment  status  others  residence  property       age  otherplans  \
0     0.052281       2       0   1.053413         3  1.057784           2   
1     0.942164       2       0   1.053413         2  0.242449           2   

   housing     cards  job    liable  tele  foreign  
0        2 -0.718745    2 -0.436436     0        0  
1        1 -0.718745    2 -0.436436     0        0  
       checkingstatus1      duration     history     purpose        amount  \
count       800.000000  8.000000e+02  800.000000  800.000000  8.000000e+02   
mean          1.580000  4.884981e-17    2.561250    3.240000  4.274359e-17   
std           1.258205  1.000626e+00    1.087532    2.755793  1.000626e+00   
min           0.000000 -1.419991e+00    0.000000    0.000000 -1.0

In [4]:
import joblib
import os

os.chdir(r"C:\Users\lenovo\Desktop\Credit Scoring Model APK\all scripts")

# Check what's in artefacts folder
for root, dirs, files in os.walk("artefacts"):
    for file in files:
        print(os.path.join(root, file))

artefacts\all_results.pkl
artefacts\best_model.pkl
artefacts\best_model_name.pkl
artefacts\best_model_summary.json
artefacts\best_params.pkl
artefacts\class_weights.pkl
artefacts\label_encoders.pkl
artefacts\scaler.pkl
artefacts\skf.pkl
artefacts\X_test.csv
artefacts\X_train.csv
artefacts\y_test.csv
artefacts\y_train.csv
artefacts\evaluation\mcnemar_results.json
artefacts\evaluation\metrics_summary.csv
artefacts\evaluation\metrics_summary.json
artefacts\evaluation\misclassified_samples.csv
artefacts\models\decision_tree\classification_report.txt
artefacts\models\decision_tree\decision_tree.pkl
artefacts\models\decision_tree\metrics.json
artefacts\models\decision_tree\params.json
artefacts\models\logistic_regression\classification_report.txt
artefacts\models\logistic_regression\logistic_regression.pkl
artefacts\models\logistic_regression\metrics.json
artefacts\models\logistic_regression\params.json
artefacts\models\random_forest\classification_report.txt
artefacts\models\random_forest\m

In [5]:
import joblib

scaler = joblib.load("artefacts/scaler.pkl")
label_encoders = joblib.load("artefacts/label_encoders.pkl")

print("Scaler feature names:", scaler.feature_names_in_.tolist())
print("\nLabel encoders keys:", list(label_encoders.keys()))

# Show categories for each encoder
for col, le in label_encoders.items():
    print(f"\n{col}: {list(le.classes_)}")

Scaler feature names: ['duration', 'amount', 'installment', 'residence', 'age', 'cards', 'liable']

Label encoders keys: ['checkingstatus1', 'history', 'purpose', 'savings', 'employ', 'status', 'others', 'property', 'otherplans', 'housing', 'job', 'tele', 'foreign']

checkingstatus1: ['A11', 'A12', 'A13', 'A14']

history: ['A30', 'A31', 'A32', 'A33', 'A34']

purpose: ['A40', 'A41', 'A410', 'A42', 'A43', 'A44', 'A45', 'A46', 'A48', 'A49']

savings: ['A61', 'A62', 'A63', 'A64', 'A65']

employ: ['A71', 'A72', 'A73', 'A74', 'A75']

status: ['A91', 'A92', 'A93', 'A94']

others: ['A101', 'A102', 'A103']

property: ['A121', 'A122', 'A123', 'A124']

otherplans: ['A141', 'A142', 'A143']

housing: ['A151', 'A152', 'A153']

job: ['A171', 'A172', 'A173', 'A174']

tele: ['A191', 'A192']

foreign: ['A201', 'A202']
